# 33 — W7 final retrain: Rank-GRPO on train+dev combined

Per RecSys_Challenge_Plan §W7 + `project_submission_prep.md`. **Phase 2** of the two-phase submission protocol: re-train the responder on train+dev combined using the W6 recipe, producing the final merged 7B for Blind-B.

## Why this stage exists

W4/W5/W6 iterated on TRAIN-ONLY with dev held out for unbiased gate evaluation. For the final Blind submission we burn dev into training (no more held-out signal — Blind-B is the next test set). This recovers the ~10–15% of training signal the dev hold-out cost us during iteration.

**Critical: this is a ONE-SHOT step.** Running it before W6's gate has passed risks burning dev into a bad model. Sequence:
1. W6 (notebook 32) trains on train-only and PASSES the §6.3 row B3 gate.
2. THEN run this notebook (33) to retrain the same recipe on train+dev.
3. Then run notebook 40 against this notebook's `MERGED_REPO`.

## Pipeline

Mirrors notebook 32 except cell 6:
1. Mount Drive, install deps, HF auth.
2. Gate-check: starting base = W6 merged (NOT W5 or W4 — W6's gate must have passed first).
3. Pytest pre-flight (W1–W7).
4. **Build train+dev parquet**: run `build_reward_dataset.py` for `--hf-split train` AND `--hf-split test`, then `build_train_plus_dev.py` to combine.
5. Augment envelope, retrieval pre-compute, build GRPO parquet.
6. TRL GRPOTrainer (G=4, scale_rewards=False) — same recipe as W6.
7. In-place merge-and-push to `recsys2026-{w7_run}-final-merged`.
8. Format-compliance smoke (sanity: format ≥ 95%; reward delta NOT measurable since dev is now in training).

## Compute
Same as W6 (~10 A100-hr) plus ~30 min retrieval pre-compute on the new train+dev rows.
Plan §W7 explicitly budgets ~28 A100-hr for the integration week — we're well inside that.

In [ ]:
# 1) GPU check.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
!git log -1 --pretty=format:'commit:  %h%nsubject: %s'

In [ ]:
# 2b) Mount Drive + persistent caches.
import os, shutil
from google.colab import drive

try: drive.mount('/content/drive')
except Exception as e:
    print(f'first mount attempt failed: {e}; retrying ...')
    try: drive.flush_and_unmount()
    except Exception: pass
    drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/recsys2026-cache'
for d in [f'{DRIVE_BASE}/hf_datasets', f'{DRIVE_BASE}/experiments_cache',
          f'{DRIVE_BASE}/grpo_final_runs']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_BASE}/hf_datasets'
%env HF_DATASETS_CACHE={DRIVE_BASE}/hf_datasets

EXPECTED_CACHE = '/content/recsys2026/music-crs-baselines/experiments/cache'
os.makedirs(os.path.dirname(EXPECTED_CACHE), exist_ok=True)
if os.path.exists(EXPECTED_CACHE) and not os.path.islink(EXPECTED_CACHE):
    shutil.rmtree(EXPECTED_CACHE)
if not os.path.islink(EXPECTED_CACHE):
    os.symlink(f'{DRIVE_BASE}/experiments_cache', EXPECTED_CACHE)

In [ ]:
# 3) HF auth — required for push_to_hub.
#
# Setup: Colab → 🔑 Secrets pane → add `HF_TOKEN` with WRITE scope.
# Get the token at https://huggingface.co/settings/tokens.
#
# Fail-fast: aborts immediately if the secret is missing or the token is
# invalid — better than failing 3 hours into training when push_to_hub fires.
import os, sys
from google.colab import userdata
from huggingface_hub import whoami

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    raise SystemExit(
        f"\u274c HF_TOKEN secret not found in Colab ({e!r}).\n"
        f"   1) Open the \U0001f511 Secrets pane in the left sidebar.\n"
        f"   2) Add a secret named exactly `HF_TOKEN` (case-sensitive).\n"
        f"   3) Toggle 'Notebook access' ON for this notebook."
    )

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

try:
    user = whoami(token=HF_TOKEN)
    HF_USERNAME = user["name"]
    os.environ["HF_USERNAME"] = HF_USERNAME
    print(f"\u2713 HF auth ok \u2014 logged in as {HF_USERNAME}")
except Exception as e:
    raise SystemExit(
        f"\u274c HF auth failed: {e!r}\n"
        f"   Token may lack WRITE scope. Regenerate at https://huggingface.co/settings/tokens"
    )

In [ ]:
# 4) Gate-check: W7 starts from the W6 MERGED base.
#
# Sequence: W4 → W5 (optional) → W6 → W7. W7 demands W6's gate PASSED.
# We READ the latest W6 gate_result.json from Drive and abort if `gate_passed`
# is False — dev burning into a bad model would be expensive.
import json
from pathlib import Path

def latest_gate(runs_dir: str):
    p = Path(runs_dir)
    if not p.exists():
        return None
    candidates = list(p.rglob('gate_result.json'))
    if not candidates:
        return None
    latest = max(candidates, key=lambda x: x.stat().st_mtime)
    with latest.open() as f:
        return json.load(f), latest

W6_RESULT = latest_gate(f'{DRIVE_BASE}/grpo_runs')
if W6_RESULT is None:
    raise SystemExit('❌ No W6 gate_result.json found. Run notebook 32 first.')
w6, w6_path = W6_RESULT
W6_MERGED = w6.get('merged_hub_model')
W6_GATE_PASSED = bool(w6.get('gate_passed'))
B1_REPO = w6.get('starting_base')  # propagated for the format-only sanity in cell 12

print(f'W6 latest: {w6_path}')
print(f'  merged repo:        {W6_MERGED}')
print(f'  gate_passed:        {W6_GATE_PASSED}')
print(f'  format compliance:  {w6.get("format_compliance_strict", 0):.1%}')
print(f'  Δ R_turn vs B1:     {w6.get("delta_r_turn_vs_b1") or "—"}')

if not W6_MERGED:
    raise SystemExit('❌ W6 gate_result.json has no merged_hub_model.')

if not W6_GATE_PASSED:
    print('\n⚠️  W6 gate did NOT pass. Burning dev into a non-passing model is risky.')
    print('    Set FORCE_W7 = True to override.')
    FORCE_W7 = False  # ← change to True ONLY if you know what you are doing.
    if not FORCE_W7:
        raise SystemExit('W7 aborted — W6 gate not passed and FORCE_W7=False.')

STARTING_MERGED = W6_MERGED
print(f'\n→ W7 starts from W6 merged base: {STARTING_MERGED}')

In [ ]:
# 5) Install deps + pytest pre-flight.
!pip install -q --upgrade transformers datasets 'pandas<3.0' tqdm omegaconf
!pip install -q --upgrade 'trl>=0.12.0' 'peft>=0.13.0' 'torchao>=0.16.0' && pip install -q flash-attn --no-build-isolation || echo 'flash-attn install failed; will fall back to SDPA at model-load time' trackio accelerate
!python -c 'import torch, transformers, trl, peft; print("torch", torch.__version__, "trl", trl.__version__, "peft", peft.__version__)'

!cd /content/recsys2026 && python -m pytest \
    tests/test_reward_fns.py \
    tests/test_state_tracker.py \
    tests/test_cmqr.py \
    tests/test_pro_rank.py \
    tests/test_augment_envelope.py \
    tests/test_build_trl_datasets.py \
    tests/test_build_sdpo_dataset.py \
    tests/test_build_grpo_dataset.py \
    tests/test_build_train_plus_dev.py \
    -q

In [ ]:
# 6) Build train+dev reward parquet (W7-specific data-prep step).
#
# Per project_submission_prep.md: iterate on TRAIN-ONLY (W4-W6); for the final
# Blind submission, retrain on TRAIN+DEV combined. This cell:
#   (a) builds reward_train.parquet via --hf-split train (~15k sessions).
#   (b) builds reward_dev.parquet  via --hf-split test (the dev set, --split-val 0.0).
#   (c) concatenates them via build_train_plus_dev.py with a `data_origin` marker.
# Idempotent: skips each step if the parquet already exists.

from pathlib import Path

N_SESSIONS_TRAIN = 15000

REWARD_TRAIN = '/content/recsys2026/data/reward_train.parquet'
REWARD_DEV   = '/content/recsys2026/data/reward_dev.parquet'
REWARD_COMBINED = '/content/recsys2026/data/reward_train_plus_dev.parquet'

if not Path(REWARD_TRAIN).exists():
    !cd /content/recsys2026 && python scripts/build_reward_dataset.py \
        --hf-split train --n-sessions {N_SESSIONS_TRAIN} \
        --split-val 0.0 \
        --out {REWARD_TRAIN}
else:
    print(f'reusing existing {REWARD_TRAIN}')

if not Path(REWARD_DEV).exists():
    # Dev set is small (~1000 sessions). --n-sessions large enough to take all of them.
    !cd /content/recsys2026 && python scripts/build_reward_dataset.py \
        --hf-split test --n-sessions 100000 \
        --split-val 0.0 \
        --out {REWARD_DEV}
else:
    print(f'reusing existing {REWARD_DEV}')

if not Path(REWARD_COMBINED).exists():
    !cd /content/recsys2026 && python scripts/build_train_plus_dev.py \
        --train {REWARD_TRAIN} --dev {REWARD_DEV} --out {REWARD_COMBINED}
else:
    print(f'reusing existing {REWARD_COMBINED}')

import pandas as pd
combined = pd.read_parquet(REWARD_COMBINED)
print(f'\ncombined train+dev rows: {len(combined):,}')
print(f'origin breakdown: {combined["data_origin"].value_counts().to_dict()}')

In [ ]:
# 7) Augment envelope on the combined train+dev parquet.
ENV_PATH = '/content/recsys2026/data/reward_train_plus_dev_envelope.parquet'

if not Path(ENV_PATH).exists():
    !cd /content/recsys2026 && python scripts/augment_envelope.py --in {REWARD_COMBINED} --out {ENV_PATH}
else:
    print(f'reusing existing {ENV_PATH}')

env_df = pd.read_parquet(ENV_PATH)
print(f'envelope-wrapped rows: {len(env_df):,}')

In [ ]:
# 8) Retrieval pre-compute (W7 train+dev rows).
#
# Mirror of notebook 32 cell 7 — same mcrs API instantiation. Only the input
# parquet differs (the combined train+dev envelope).
import json, sys, re
import pandas as pd
from tqdm import tqdm
import torch

RETRIEVAL_OUT = '/content/recsys2026/data/trl/grpo_final_retrieval.parquet'
RETRIEVAL_PARTIAL = '/content/recsys2026/data/trl/grpo_final_retrieval.partial.parquet'

if Path(RETRIEVAL_OUT).exists():
    print(f'reusing existing {RETRIEVAL_OUT}')
else:
    sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
    from mcrs.retrieval_modules import load_retrieval_module
    from mcrs.rerankers.pro_rank import ProRankReranker
    from mcrs.query_rewriters.cmqr import CMQR_REWRITER
    from mcrs.query_rewriters.state_tracker import StateTracker
    from mcrs.lm_modules import load_lm_module
    from mcrs.db_item import MusicCatalogDB

    ITEM_DB     = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
    DATASET     = 'talkpl-ai/TalkPlayData-Challenge-Dataset'
    SPLITS      = ['all_tracks']
    CORPUS      = ['track_name', 'artist_name', 'album_name']
    CACHE_DIR   = '/content/recsys2026/music-crs-baselines/experiments/cache'
    LM_TYPE     = 'meta-llama/Llama-3.2-1B-Instruct'
    PROMPTS_DIR = '/content/recsys2026/music-crs-baselines/mcrs/system_prompts'

    print('[w7-retrieval] loading LM…')
    lm = load_lm_module(lm_type=LM_TYPE, device='cuda', attn_implementation='sdpa',
                       dtype=torch.bfloat16, use_vllm=False)
    print('[w7-retrieval] loading wRRF…')
    retrieval = load_retrieval_module('wrrf_bm25_dense_lyrics_v1', ITEM_DB, SPLITS, CORPUS, CACHE_DIR)
    print('[w7-retrieval] state-tracker + CMQR…')
    state_tracker = StateTracker(lm=lm, prompt_path=f'{PROMPTS_DIR}/state_extraction.txt',
                                cache_dir=CACHE_DIR, max_new_tokens=96)
    cmqr = CMQR_REWRITER(lm=lm, inner_retriever=retrieval,
                        prompt_path=f'{PROMPTS_DIR}/cmqr_rewrites.txt',
                        cache_dir=CACHE_DIR,
                        n_rewrites=4, topk_per_rewrite=50, rrf_k=60, max_new_tokens=96)
    print('[w7-retrieval] ProRank with rationales…')
    reranker = ProRankReranker(item_db_name=ITEM_DB, track_split_types=SPLITS,
                              corpus_types=CORPUS, cache_dir=CACHE_DIR,
                              with_rationales=True)
    item_db = MusicCatalogDB(dataset_name=ITEM_DB, split_types=SPLITS, corpus_types=CORPUS)
    valid_catalog = set(item_db.metadata_dict.keys())

    pos_df = env_df.query('label == 1').drop_duplicates(['session_id', 'turn_number'])
    print(f'unique POS turns to retrieve: {len(pos_df):,}')

    # Gold tracks + per-turn user_query/history from BOTH train and test HF splits.
    from datasets import load_dataset
    sid2turn_to_gold = {}
    sid2turn_to_user_query = {}
    sid2turn_to_history_text = {}
    for hf_split in ['train', 'test']:
        print(f'[w7-retrieval] indexing HF split={hf_split}…')
        raw = load_dataset(DATASET, split=hf_split)
        for sess in raw:
            sid = sess['session_id']
            msgs = sorted(sess['conversations'], key=lambda m: (int(m['turn_number']), m['role']))
            for msg in msgs:
                if msg['role'] == 'music':
                    sid2turn_to_gold[(sid, int(msg['turn_number']))] = str(msg['content'])
            for tn in range(1, 9):
                prior = [m for m in msgs if int(m['turn_number']) < tn]
                history = '\n'.join(f"{m['role']}: {m['content']}" for m in prior)
                user_msg = next((m for m in msgs if int(m['turn_number']) == tn and m['role'] == 'user'), None)
                if user_msg is None:
                    continue
                sid2turn_to_user_query[(sid, tn)] = str(user_msg['content'])
                sid2turn_to_history_text[(sid, tn)] = history

    # Resume from partial.
    partial_rows = []
    if Path(RETRIEVAL_PARTIAL).exists():
        partial_rows = pd.read_parquet(RETRIEVAL_PARTIAL).to_dict('records')
        partial_df = pd.DataFrame(partial_rows).drop_duplicates(['session_id', 'turn_number'], keep='last')
        partial_rows = partial_df.to_dict('records')
        done_keys = {(r['session_id'], int(r['turn_number'])) for r in partial_rows}
        pos_df = pos_df[~pos_df.apply(lambda r: (r['session_id'], int(r['turn_number'])) in done_keys, axis=1)]
        print(f'resuming — {len(done_keys):,} done; remaining {len(pos_df):,}')

    rows_out = list(partial_rows)
    BATCH = 16
    rows_buffer = []
    for i, row in enumerate(tqdm(pos_df.itertuples(index=False), total=len(pos_df), desc='retrieve')):
        rows_buffer.append((row.session_id, int(row.turn_number)))
        if len(rows_buffer) < BATCH and i + 1 < len(pos_df):
            continue
        sids = [s for s, _ in rows_buffer]
        tns  = [t for _, t in rows_buffer]
        queries = [sid2turn_to_user_query.get((s, t), '') for s, t in rows_buffer]
        histories = [sid2turn_to_history_text.get((s, t), '') for s, t in rows_buffer]
        states = []
        for s, t, q, h in zip(sids, tns, queries, histories):
            try: states.append(state_tracker.extract(s, t, q, h))
            except Exception: states.append(None)
        cmqr.set_batch_context(session_ids=sids, turn_numbers=tns, extracted_states=states)
        try:
            top100 = cmqr.batch_text_to_item_retrieval(queries, topk=100, user_ids=[None] * len(queries))
        except TypeError:
            top100 = cmqr.batch_text_to_item_retrieval(queries, topk=100)
        top20 = reranker.rerank(queries, top100, topk=20)
        for s, t, q, ids20, pool100 in zip(sids, tns, queries, top20, top100):
            seen, kept = set(), []
            for tid in ids20:
                if tid in seen or tid not in valid_catalog: continue
                kept.append(tid); seen.add(tid)
            if len(kept) < 20:
                for tid in pool100:
                    if len(kept) >= 20: break
                    if tid in seen or tid not in valid_catalog: continue
                    kept.append(tid); seen.add(tid)
            kept = kept[:20]
            try: rationales = reranker.generate_rationales(q, kept)
            except Exception: rationales = ['' for _ in kept]
            top1_tid = kept[0] if kept else ''
            top1_meta = item_db.metadata_dict.get(top1_tid, {}) if top1_tid else {}
            tn_get = lambda field: (top1_meta.get(field) or [''])
            top1_track_name  = (tn_get('track_name')[0] if isinstance(tn_get('track_name'), list) else str(tn_get('track_name'))) or ''
            top1_artist_name = (tn_get('artist_name')[0] if isinstance(tn_get('artist_name'), list) else str(tn_get('artist_name'))) or ''
            rows_out.append({
                'session_id': s, 'turn_number': t,
                'gold_track_id': sid2turn_to_gold.get((s, t), ''),
                'predicted_track_ids': kept,
                'top1_track_name': top1_track_name,
                'top1_artist_name': top1_artist_name,
                'reranker_rationales': rationales,
            })
        rows_buffer.clear()
        if (i + 1) % 500 == 0:
            partial_out = pd.DataFrame(rows_out).drop_duplicates(['session_id', 'turn_number'], keep='last')
            partial_out.to_parquet(RETRIEVAL_PARTIAL, index=False)

    out_df = pd.DataFrame(rows_out).drop_duplicates(['session_id', 'turn_number'], keep='last')
    Path(RETRIEVAL_OUT).parent.mkdir(parents=True, exist_ok=True)
    out_df.to_parquet(RETRIEVAL_OUT, index=False)
    if Path(RETRIEVAL_PARTIAL).exists(): Path(RETRIEVAL_PARTIAL).unlink()
    print(f'\n✓ retrieval cache: {len(out_df):,} rows → {RETRIEVAL_OUT}')

ret_df = pd.read_parquet(RETRIEVAL_OUT)
print(f'retrieval rows: {len(ret_df):,}')

In [ ]:
# 9) Build the GRPO parquet (train+dev envelope × retrieval cache).
GRPO_OUT = '/content/recsys2026/data/trl/grpo_final.parquet'
SYSTEM_PROMPT_TXT = '/content/recsys2026/data/trl/grpo_system_prompt.txt'

PROMPTS_DIR = '/content/recsys2026/music-crs-baselines/mcrs/system_prompts'
with open(f'{PROMPTS_DIR}/roleplay.txt', encoding='utf-8') as f: role_play = f.read()
with open(f'{PROMPTS_DIR}/response_generation_cot_user_state.txt', encoding='utf-8') as f: cot_prompt = f.read()
SYSTEM_PROMPT_STR = role_play + '\n\n' + cot_prompt
Path(SYSTEM_PROMPT_TXT).parent.mkdir(parents=True, exist_ok=True)
with open(SYSTEM_PROMPT_TXT, 'w', encoding='utf-8') as f: f.write(SYSTEM_PROMPT_STR)

if not Path(GRPO_OUT).exists():
    !cd /content/recsys2026 && python scripts/build_grpo_dataset.py \
        --envelope {ENV_PATH} \
        --retrieval {RETRIEVAL_OUT} \
        --system-prompt-path {SYSTEM_PROMPT_TXT} \
        --out {GRPO_OUT}

import pandas as pd, numpy as np
d = pd.read_parquet(GRPO_OUT)
print(f'\nGRPO final dataset: {len(d):,} rows')
print(f'splits: {d["split"].value_counts().to_dict()}')

n_pos_total = pd.read_parquet(ENV_PATH).query('label == 1').shape[0]
unjoined_frac = (n_pos_total - len(d)) / max(n_pos_total, 1)
print(f'POS coverage: {len(d):,} / {n_pos_total:,} ({100*(1-unjoined_frac):.0f}%)')
if unjoined_frac > 0.30:
    raise SystemExit(f'❌ {100*unjoined_frac:.0f}% of POS turns missing from retrieval; rerun cell 8.')

In [ ]:
# 10) Schema validation + tiny eval slice.
#
# DESIGN NOTE: dev is now in TRAINING. There is no honest held-out slice for the
# §6.3 row B3 +0.03 reward gate. We carve a tiny 50-row slice from the combined
# data purely as a TRAINING-TIME monitor for `frac_reward_zero_std` and rough
# format compliance. DO NOT interpret as a generalization gate.
from datasets import Dataset
import numpy as np

def _normalize_prompt(p):
    if isinstance(p, np.ndarray): return [dict(m) for m in p]
    if isinstance(p, list) and p and isinstance(p[0], np.ndarray): return [dict(m) for m in p]
    return p
d['prompt'] = d['prompt'].apply(_normalize_prompt)
ds = Dataset.from_pandas(d, preserve_index=False)
required = {'prompt', 'gold_track_id', 'predicted_track_ids',
            'top1_meta_json', 'user_state_json', 'history_text', 'split'}
assert required.issubset(set(ds.column_names))
print(f'✓ schema PASS — columns: {ds.column_names}')

# 99/1 split: most goes to training; tiny 1% slice is just for monitor.
split = ds.train_test_split(test_size=0.01, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(f'train: {len(train_ds):,}  monitor-eval: {len(eval_ds):,}')

In [ ]:
# 11) Trackio init.
from datetime import date
import trackio
from transformers import TrainerCallback

RUN_NAME = f'b3-grpo-final-{date.today().isoformat()}'
TRACKIO_OK = True
try:
    trackio.init(
        project='recsys2026', name=RUN_NAME, group='b-stage',
        config={
            'model': STARTING_MERGED, 'method': 'W7 final retrain (Rank-GRPO on train+dev)',
            'lora_r': 32, 'num_generations': 4, 'scale_rewards': False,
            'kl_beta': 0.04, 'lr': 5e-6, 'max_steps': 12_000,
        },
    )
    print(f'✓ Trackio run: {RUN_NAME}')
except Exception as e:
    TRACKIO_OK = False
    print(f'⚠️  Trackio init failed: {e!r}')

class TrackioCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not TRACKIO_OK or not logs: return
        try: trackio.log({k: float(v) for k, v in logs.items() if isinstance(v, (int, float))})
        except Exception: pass

In [ ]:
# 12) Rank-GRPO training — Qwen-3B + W6-merged base + fresh LoRA r=32.
#
# 2026-05-03 model-size switch (7B → 3B): STARTING_MERGED auto-resolves
# from W6 gate_result.json so it inherits whatever upstream produced.
# Batch settings tightened for 3B's smaller VRAM footprint:
#   per_device_train_batch_size: 1 → 2 (G=4 × bs=2 = 8 trajectories/micro-batch
#     fits comfortably on A100-40GB with 3B)
#   gradient_accumulation_steps: 4 → 2 (effective batch unchanged at 4)
# grad_ckpt stays True per recipe.
#
# Deep-review P0-1 fix: this cell now wires `DistilledJudge` (Option B
# refactor). Without this, R_judge contributed 0 gradient at training and
# 30% of the reward weight was inert.
#
# JUDGE_HUB_REPO resolution: prefer an explicit hardcode (after running
# colab/31p), else read from the latest judge_runs/ gate. If neither
# exists, judge falls back to stub (returns 0.0) — RUN AT YOUR OWN RISK.
import json as _json
from pathlib import Path as _P
import os as _os

JUDGE_HUB_REPO = _os.environ.get('JUDGE_HUB_REPO', None)
if JUDGE_HUB_REPO is None:
    # Fallback: try the latest pilot run's gate_result.json.
    pilot_runs = _P(f'{DRIVE_BASE}/grpo_pilot_runs')
    candidates = list(pilot_runs.rglob('gate_result.json')) if pilot_runs.exists() else []
    if candidates:
        latest = max(candidates, key=lambda x: x.stat().st_mtime)
        with latest.open() as _f:
            _gate = _json.load(_f)
        JUDGE_HUB_REPO = _gate.get('judge')

if JUDGE_HUB_REPO is None:
    print('⚠️  No JUDGE_HUB_REPO set and no pilot gate found. R_judge will be 0.0.')
    print('   Set env JUDGE_HUB_REPO=<repo> or run colab/31p first to train + record one.')
else:
    print(f'✓ Using distilled judge: {JUDGE_HUB_REPO}')

import torch, gc, json, sys
from peft import LoraConfig
from trl import GRPOTrainer, GRPOConfig

sys.path.insert(0, '/content/recsys2026/scripts')
from reward_fns import compose_r_turn, r_format, DistilledJudge

# Instantiate + warm up the judge before trainer init (P1-4 fix).
JUDGE = DistilledJudge(checkpoint=JUDGE_HUB_REPO)
JUDGE.warmup()  # forces Hub download + first forward NOW, not on step 1

HUB_REPO = f'{HF_USERNAME}/recsys2026-{RUN_NAME}'
OUTPUT_DIR = f'/content/recsys2026/training_runs/{RUN_NAME}'

peft_config = LoraConfig(
    r=32, lora_alpha=32, lora_dropout=0.05,
    bias='none', task_type='CAUSAL_LM', target_modules='all-linear',
)


def _completion_text(completion):
    if isinstance(completion, list) and completion:
        last = completion[-1]
        if isinstance(last, dict):
            return str(last.get('content', ''))
        return str(last)
    return str(completion)


def _user_content_from_prompt(prompt):
    """Extract the user-role content for the judge's `context` arg."""
    if isinstance(prompt, list):
        for msg in prompt:
            if isinstance(msg, dict) and msg.get('role') == 'user':
                return str(msg.get('content', ''))
        return ''
    return str(prompt)


def reward_main(prompts, completions, **kwargs):
    """Composite Option B reward (deep-review P0-1: judge + group bonus wired).
    Format gate stays in the separate reward_format fn so a bad rollout
    doesn't multiplicatively zero the gradient.
    """
    from collections import defaultdict
    groups = defaultdict(list)
    sids = kwargs['session_id']; tns = kwargs['turn_number']
    for i in range(len(completions)):
        groups[(sids[i], tns[i])].append(i)

    contexts = [_user_content_from_prompt(p) for p in prompts]
    completion_texts = [_completion_text(c) for c in completions]
    judge_scores = JUDGE.score_batch(contexts, completion_texts, batch_size=16)

    scores = []
    for i, completion in enumerate(completions):
        peer_indices = groups[(sids[i], tns[i])]
        peer_responses = [completion_texts[j] for j in peer_indices]
        comps = compose_r_turn(
            predicted_track_ids=list(kwargs['predicted_track_ids'][i]),
            gold_track_id=kwargs['gold_track_id'][i],
            response_text=completion_texts[i],
            valid_catalog=None,
            top1_meta=json.loads(kwargs['top1_meta_json'][i]),
            user_state=json.loads(kwargs['user_state_json'][i]),
            user_profile=json.loads(kwargs['user_profile_json'][i]) if 'user_profile_json' in kwargs else None,  # gap-analysis Step 3: user_profile piped
            history_text=kwargs['history_text'][i],
            judge_score=judge_scores[i],
            include_format=False,
            group_responses=peer_responses,
        )
        scores.append(comps['r_turn'])
    return scores


def reward_format(prompts, completions, **kwargs):
    return [r_format(_completion_text(c)) for c in completions]


config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    model_init_kwargs={"torch_dtype": "bfloat16", "attn_implementation": "flash_attention_2"},  # FA2 ~30% speedup; if unavailable TRL falls back automatically
    push_to_hub=True, hub_model_id=HUB_REPO, hub_strategy='every_save', hub_private_repo=True,
    num_generations=4,
    scale_rewards=False,
    max_completion_length=256,  # was 320 → 256 (~20% rollout speedup)
    temperature=0.9, beta=0.04,
    reward_weights=[0.95, 0.05],
    max_steps=12_000,
    per_device_train_batch_size=2, gradient_accumulation_steps=2,  # was 1×4 (3B unlocks 2×2)
    learning_rate=5e-6, lr_scheduler_type='cosine', warmup_ratio=0.05,
    bf16=True, gradient_checkpointing=True,
    eval_strategy='no',  # speedup: skip mid-training eval eval_steps=500, per_device_eval_batch_size=2,
    save_strategy='steps', save_steps=250, save_total_limit=3,
    logging_steps=20, report_to='none',
    seed=42, data_seed=42,  # explicit seeds — reproducibility across re-runs
)

callbacks = [TrackioCallback()] if TRACKIO_OK else []

# PRIOR_STAGE was set in cell 5 (W6 gate-check). Reuse if available; else label.
try:
    _prior = PRIOR_STAGE  # noqa
except NameError:
    PRIOR_STAGE = 'W6'

trainer = GRPOTrainer(
    model=STARTING_MERGED, args=config,
    train_dataset=train_ds, eval_dataset=eval_ds,
    reward_funcs=[reward_main, reward_format],
    peft_config=peft_config, callbacks=callbacks,
)

print(f'🚀 W7 final retrain (Rank-GRPO + DistilledJudge) on train+dev (~5 A100-hr on 3B; was ~10 on 7B)...')
print(f'   model:   {STARTING_MERGED}  ({PRIOR_STAGE}-merged base)')
print(f'   judge:   {JUDGE_HUB_REPO or "STUB (R_judge=0)"}')
print(f'   adapter → {HUB_REPO}')
print(f'   G={config.num_generations}  scale_rewards={config.scale_rewards}  '
      f'effective_bs={config.per_device_train_batch_size * config.gradient_accumulation_steps} × G')

# Auto-resume if a checkpoint already exists in OUTPUT_DIR (idempotent re-run).
from pathlib import Path as _Path
_has_ckpt = any(_Path(OUTPUT_DIR).glob('checkpoint-*'))
if _has_ckpt:
    print(f'   ↻ found existing checkpoint(s) under {OUTPUT_DIR} — resuming')
trainer.train(resume_from_checkpoint=_has_ckpt)
print('✓ training complete')

In [ ]:
# 13) Push adapter + Drive backup.
trainer.push_to_hub()
import shutil
drive_dst = f'{DRIVE_BASE}/grpo_final_runs/{RUN_NAME}'
shutil.copytree(OUTPUT_DIR, drive_dst, dirs_exist_ok=True)
print(f'✓ adapter at https://huggingface.co/{HUB_REPO}')
print(f'✓ Drive mirror at {drive_dst}')

In [ ]:
# 14) In-place merge → push final deployment artifact.
import gc, torch
from transformers import AutoTokenizer

trainer.optimizer = None; trainer.lr_scheduler = None
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
print(f'free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

print(f'merging W7 LoRA into {STARTING_MERGED} (W6-merged base) in-place…')
fully_merged = trainer.model.merge_and_unload()
tok = AutoTokenizer.from_pretrained(STARTING_MERGED)

MERGED_REPO = f'{HF_USERNAME}/recsys2026-{RUN_NAME}-merged'
print(f'pushing → {MERGED_REPO}')
fully_merged.push_to_hub(MERGED_REPO, private=True,
                         commit_message=f'W7 final (train+dev) GRPO merged on {STARTING_MERGED}')
tok.push_to_hub(MERGED_REPO, private=True)
print(f'✓ DEPLOYMENT ARTIFACT at https://huggingface.co/{MERGED_REPO}')
print(f'  → set lm_type in config/300-final-blindset-B.yaml to this repo, leave lora_path: null')

In [ ]:
# 15) Format-compliance sanity (training-time monitor only — NOT a generalization gate).
#
# Dev is now in training; the +0.03 R_turn gate is no longer measurable here.
# What we DO check: the W7 model still emits the envelope with ≥95% strict
# compliance. A regression here means GRPO destabilized format fluency.
import json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import sys
sys.path.insert(0, '/content/recsys2026/scripts')
from reward_fns import r_format, ENVELOPE

w7_tok = AutoTokenizer.from_pretrained(MERGED_REPO)
w7_model = AutoModelForCausalLM.from_pretrained(MERGED_REPO, torch_dtype=torch.bfloat16, device_map='cuda').eval()

def gen(model, tok, conv):
    formatted = tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
    enc = tok(formatted, return_tensors='pt', truncation=True, max_length=2048).to('cuda')
    with torch.no_grad():
        out_ids = model.generate(**enc, max_new_tokens=320, do_sample=False,
                                 pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out_ids[0, enc['input_ids'].shape[1]:], skip_special_tokens=True)

n_eval = min(50, len(eval_ds))
n_strict = n_loose = 0
samples = []
for i in range(n_eval):
    out = gen(w7_model, w7_tok, eval_ds[i]['prompt'])
    if r_format(out) == 1.0: n_strict += 1
    if ENVELOPE.search(out): n_loose += 1
    if len(samples) < 3: samples.append(out[:400])

compliance_strict = n_strict / n_eval
compliance_loose = n_loose / n_eval
print(f'\nFORMAT (post-W7):')
print(f'  strict: {n_strict}/{n_eval} = {compliance_strict:.1%}')
print(f'  loose:  {n_loose}/{n_eval} = {compliance_loose:.1%}')

print('\nSamples:')
for i, s in enumerate(samples, 1):
    print(f'\n--- {i} ---\n{s}')

# Persist gate result.
import os
from datetime import date
result = {
    'stage': 'W7-final',
    'run_name': RUN_NAME,
    'date': date.today().isoformat(),
    'starting_base': STARTING_MERGED,
    'hub_model': HUB_REPO,
    'merged_hub_model': MERGED_REPO,
    'format_compliance_strict': compliance_strict,
    'format_compliance_loose': compliance_loose,
    'gate_passed': compliance_strict >= 0.95,
    'note': 'Dev is in training; reward delta vs B1 not measurable here. '
            'See colab/40 for Blind-B inference + dev nDCG@20 no-regression smoke.',
}
out_path = f'{DRIVE_BASE}/grpo_final_runs/{RUN_NAME}/gate_result.json'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, 'w', encoding='utf-8') as f: json.dump(result, f, ensure_ascii=False, indent=2)
print(f'\ngate_result → {out_path}')

if TRACKIO_OK:
    trackio.log({'format_compliance_strict': compliance_strict,
                 'format_compliance_loose': compliance_loose})
    trackio.finish()

## Next step

After this notebook completes:
1. Update `music-crs-baselines/config/300-final-blindset-B.yaml` — replace `PLACEHOLDER` with the run-name produced here. The `MERGED_REPO` from cell 14 is the value to paste.
2. Run `colab/40_run_blindset_B.ipynb` — runs `run_inference_blindset.py` against this merged repo, packages with `validate_prediction.py`, and produces the CodaBench zip.
3. Upload the zip via the CodaBench portal (≤1/week per plan §2.6).

If format compliance regressed below 95%, treat as a W7 failure: revert to W6's MERGED_REPO for the Blind-B submission and skip the dev-burn this round.